In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip uninstall -y pillow torchao -q
!pip install -q pillow==11.2.1
!pip install -q transformers==4.57.1 peft==0.18.0 accelerate==1.11.0 bitsandbytes==0.48.1 sentencepiece

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import gc, ast, json, time, random, math
import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import LoraConfig, get_peft_model, PeftModel, TaskType

ROOT = Path("/content/drive/MyDrive/pixels-to-predictions")

TRAIN_CSV = ROOT / "train.csv"
VAL_CSV = ROOT / "val.csv"
TEST_CSV = ROOT / "test.csv"
IMG_ROOT = ROOT / "images" / "images"

BASE_MODEL_NAME = "HuggingFaceTB/SmolVLM-500M-Instruct"

OUT_DIR = ROOT / "outputs_smolvlm_dpo_ipo_cached_ref_t4"
BEST_DIR = OUT_DIR / "best_adapter"
LAST_DIR = OUT_DIR / "last_adapter"
REF_CACHE_PATH = OUT_DIR / "train_ref_yes_logprobs.csv"

SUB_PATH = ROOT / "submission_dpo_cached_ref.csv"
DEBUG_PATH = ROOT / "submission_dpo_cached_ref_debug.csv"

LETTERS = ["A", "B", "C", "D", "E"]

SEED = 42
MAX_IMAGE_SIZE = 384

EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 8
NUM_WORKERS = 2
PRINT_EVERY = 50

LR = 5.0e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.05

LOSS_TYPE = "dpo"
BETA = 0.1
MAX_REJECTED_PER_Q = 2

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32


assert TRAIN_CSV.exists(), TRAIN_CSV
assert VAL_CSV.exists(), VAL_CSV
assert TEST_CSV.exists(), TEST_CSV
assert REF_CACHE_PATH.exists(), f"Missing cached ref scores: {REF_CACHE_PATH}"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

train_ref_df = pd.read_csv(REF_CACHE_PATH)

ref_lookup = {}
for _, r in train_ref_df.iterrows():
    ref_lookup[(int(r["row_idx"]), int(r["choice_idx"]))] = float(r["ref_yes_logprob"])

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
print("cached ref rows:", len(train_ref_df))
print("ref lookup size:", len(ref_lookup))

def parse_choices(choices):
    if choices is None:
        return []
    if isinstance(choices, list):
        return choices
    if isinstance(choices, dict):
        return [choices[k] for k in LETTERS if k in choices]

    s = str(choices).strip()
    if s.lower() in ["", "nan", "none"]:
        return []

    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            return [parsed[k] for k in LETTERS if k in parsed]
    except Exception:
        pass

    try:
        parsed = json.loads(s)
        if isinstance(parsed, list):
            return parsed
        if isinstance(parsed, dict):
            return [parsed[k] for k in LETTERS if k in parsed]
    except Exception:
        pass

    return [s]


def find_image(row, split):
    raw = str(row.get("image_path", "")).strip()
    qid = str(row.get("id", "")).strip()

    candidates = []

    if raw and raw.lower() not in ["", "nan", "none"]:
        raw_path = Path(raw)
        candidates.extend([
            ROOT / raw,
            ROOT / "images" / raw_path.name,
            ROOT / "images" / split / raw_path.name,
            ROOT / "images" / "images" / raw_path.name,
            ROOT / "images" / "images" / split / raw_path.name,
            IMG_ROOT / split / raw_path.name,
        ])

    if qid and qid.lower() not in ["", "nan", "none"]:
        for ext in [".png", ".jpg", ".jpeg", ".webp"]:
            candidates.extend([
                IMG_ROOT / split / f"{qid}{ext}",
                ROOT / "images" / "images" / split / f"{qid}{ext}",
                ROOT / "images" / split / f"{qid}{ext}",
            ])

    for c in candidates:
        if c.exists():
            return c

    return None


def is_image_heavy(row):
    text = " ".join([
        str(row.get("question", "")),
        str(row.get("skill", "")),
        str(row.get("topic", "")),
        str(row.get("category", "")),
        str(row.get("hint", "")),
        str(row.get("lecture", "")),
    ]).lower()

    keys = [
        "image", "picture", "diagram", "graph", "chart", "table", "map",
        "model", "shown", "figure", "punnett", "square", "label", "labels",
        "axis", "axes", "food web", "circuit", "rock cycle", "life cycle",
        "visible", "below", "above", "shown below", "the diagram"
    ]

    return any(k in text for k in keys)


def preprocess_image(img, row=None):
    img = img.convert("RGB")
    w, h = img.size

    scale = min(MAX_IMAGE_SIZE / max(w, h), 1.0)
    if scale < 1.0:
        img = img.resize((int(w * scale), int(h * scale)), Image.Resampling.BICUBIC)

    if row is not None and is_image_heavy(row):
        img = ImageEnhance.Contrast(img).enhance(1.45)
        img = ImageEnhance.Sharpness(img).enhance(1.7)

    return img


def load_image(row, split):
    img_path = find_image(row, split)

    if img_path is None:
        return Image.new("RGB", (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), "white"), None

    try:
        img = Image.open(img_path).convert("RGB")
        img = preprocess_image(img, row=row)
        return img, img_path
    except Exception:
        return Image.new("RGB", (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), "white"), None


def detect_question_type(row):
    text = " ".join([
        str(row.get("question", "")),
        str(row.get("skill", "")),
        str(row.get("topic", "")),
        str(row.get("category", "")),
        str(row.get("hint", "")),
        str(row.get("lecture", "")),
    ]).lower()

    if "punnett" in text or "homozygous" in text or "heterozygous" in text or "offspring" in text or "genotype" in text or "allele" in text:
        return "punnett"
    if "graph" in text or "chart" in text or "axis" in text or "axes" in text or "plot" in text:
        return "graph"
    if "table" in text or "row" in text or "column" in text:
        return "table"
    if "map" in text or "region" in text or "location" in text:
        return "map"
    if "food web" in text or "food chain" in text or "producer" in text or "consumer" in text:
        return "food_web"
    if "circuit" in text or "electric" in text or "battery" in text or "bulb" in text:
        return "circuit"
    if "passage" in text or "read the text" in text or "informational text" in text or "reading-comprehension" in text:
        return "passage"
    if "diagram" in text or "model" in text or "shown" in text or "figure" in text or "label" in text:
        return "diagram"
    return "general"


def type_guidance(qtype):
    if qtype == "punnett":
        return (
            "This may involve a Punnett square or genetics. Use the four boxes if shown. "
            "Identify genotypes like AA, Aa, or aa. Homozygous dominant means two uppercase alleles. "
            "Homozygous recessive means two lowercase alleles. Heterozygous means one uppercase and one lowercase. "
            "Dominant phenotype usually means at least one uppercase allele. Recessive phenotype usually means two lowercase alleles. "
            "For probability, count matching boxes out of four. For ratios, follow the order stated in the question."
        )
    if qtype == "graph":
        return "Read the graph title, axes, units, labels, legend, values, and trend carefully."
    if qtype == "table":
        return "Read row labels, column labels, and exact cell values carefully."
    if qtype == "map":
        return "Use labels, legend, directions, relative positions, regions, and locations."
    if qtype == "food_web":
        return "Follow arrow direction carefully. Energy usually flows from food/prey to consumer/predator."
    if qtype == "circuit":
        return "Trace whether the circuit is open or closed. Use batteries, wires, switches, bulbs, and component positions."
    if qtype == "passage":
        return "Use the passage/hint evidence carefully. Choose the answer directly supported by the text."
    if qtype == "diagram":
        return "Read labels, arrows, symbols, object positions, and relationships in the diagram."
    return "Use the image if useful, plus the question, choices, lecture, hint, and metadata."


def build_pref_prompt(row, choice_text):
    question = str(row.get("question", "")).strip()
    subject = str(row.get("subject", "")).strip()
    topic = str(row.get("topic", "")).strip()
    category = str(row.get("category", "")).strip()
    skill = str(row.get("skill", "")).strip()
    lecture = str(row.get("lecture", "")).strip()
    hint = str(row.get("hint", "")).strip()

    qtype = detect_question_type(row)
    guide = type_guidance(qtype)

    return f"""
You are judging whether an answer choice is correct for a science multiple-choice question.

Use the image, question, lecture, hint, skill, and metadata.
Decide if the proposed answer is the best supported answer.

Question type guidance:
{guide}

Question:
{question}

Subject: {subject}
Topic: {topic}
Category: {category}
Skill: {skill}

Lecture:
{lecture}

Hint:
{hint}

Proposed answer:
{choice_text}

Is this proposed answer correct?

Answer only Yes or No.
""".strip()


processor = AutoProcessor.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)

def get_token_ids_for_words(words):
    ids = []
    for w in words:
        toks = processor.tokenizer(w, add_special_tokens=False)["input_ids"]
        if toks:
            ids.append(toks[-1])
    return list(set(ids))

YES_TOKEN_IDS = get_token_ids_for_words(["Yes", " yes", "Yes.", " yes."])
NO_TOKEN_IDS = get_token_ids_for_words(["No", " no", "No.", " no."])

print("YES_TOKEN_IDS:", YES_TOKEN_IDS)
print("NO_TOKEN_IDS:", NO_TOKEN_IDS)

def make_chat_text(prompt):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": str(prompt).replace("<image>", "").strip()},
            ],
        }
    ]
    return processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False
    )


def yes_logprob_from_logits(logits):
    log_probs = F.log_softmax(logits.float(), dim=-1)
    return torch.logsumexp(log_probs[:, YES_TOKEN_IDS], dim=1)


base_policy = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

target_modules = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_policy, lora_config)
model.train()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")
assert trainable <= 5_000_000, f"Too many trainable params: {trainable:,}"
model.print_trainable_parameters()


class PreferenceMCQDataset(Dataset):
    def __init__(self, df, split):
        self.df = df.reset_index(drop=True)
        self.split = split

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image, img_path = load_image(row, self.split)

        choices = parse_choices(row.get("choices", ""))
        answer_idx = int(row["answer"])

        valid_wrong = [i for i in range(len(choices)) if i != answer_idx]

        if len(valid_wrong) > MAX_REJECTED_PER_Q:
            random.shuffle(valid_wrong)
            valid_wrong = valid_wrong[:MAX_REJECTED_PER_Q]

        chosen_letter = LETTERS[answer_idx]
        chosen_text = f"{chosen_letter}. {str(choices[answer_idx]).strip()}"
        chosen_prompt = build_pref_prompt(row, chosen_text)

        rejected_prompts = []
        rejected_indices = []

        for wi in valid_wrong:
            letter = LETTERS[wi]
            choice_text = f"{letter}. {str(choices[wi]).strip()}"
            rejected_prompts.append(build_pref_prompt(row, choice_text))
            rejected_indices.append(wi)

        return {
            "row_idx": idx,
            "id": row.get("id", idx),
            "image": image,
            "chosen_prompt": chosen_prompt,
            "rejected_prompts": rejected_prompts,
            "answer_idx": answer_idx,
            "rejected_indices": rejected_indices,
        }


def collate_pref(batch):
    item = batch[0]
    image = item["image"]

    texts = []
    images = []
    choice_indices = []

    chosen_text = make_chat_text(item["chosen_prompt"])
    rejected_texts = [make_chat_text(p) for p in item["rejected_prompts"]]

    texts.append(chosen_text)
    images.append(image)
    choice_indices.append(item["answer_idx"])

    for rt, wi in zip(rejected_texts, item["rejected_indices"]):
        texts.append(rt)
        images.append(image)
        choice_indices.append(wi)

    pair_map = []
    for j in range(1, len(texts)):
        pair_map.append((0, j))

    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    return {
        "inputs": inputs,
        "pair_map": pair_map,
        "row_idx": int(item["row_idx"]),
        "choice_indices": choice_indices,
    }


train_ds = PreferenceMCQDataset(train_df, "train")

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_pref,
    pin_memory=True,
)


def preference_loss(batch):
    inputs = batch["inputs"]

    model_device = next(model.parameters()).device
    inputs = {
        k: v.to(model_device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    outputs = model(**inputs)
    logits = outputs.logits[:, -1, :]

    policy_yes_lp = yes_logprob_from_logits(logits)

    row_idx = int(batch["row_idx"])
    choice_indices = batch["choice_indices"]

    ref_yes_lp_values = []
    for choice_idx in choice_indices:
        ref_yes_lp_values.append(ref_lookup[(row_idx, int(choice_idx))])

    ref_yes_lp = torch.tensor(
        ref_yes_lp_values,
        dtype=policy_yes_lp.dtype,
        device=policy_yes_lp.device
    )

    losses = []

    for chosen_i, rejected_i in batch["pair_map"]:
        pi_lograt = policy_yes_lp[chosen_i] - policy_yes_lp[rejected_i]
        ref_lograt = ref_yes_lp[chosen_i] - ref_yes_lp[rejected_i]

        logits_pref = pi_lograt - ref_lograt

        if LOSS_TYPE.lower() == "dpo":
            loss = -F.logsigmoid(BETA * logits_pref)
        elif LOSS_TYPE.lower() == "ipo":
            target = 1.0 / (2.0 * BETA)
            loss = (logits_pref - target) ** 2
        else:
            raise ValueError("LOSS_TYPE must be 'dpo' or 'ipo'")

        losses.append(loss)

    if len(losses) == 0:
        return policy_yes_lp.sum() * 0.0

    return torch.stack(losses).mean()


@torch.no_grad()
def score_row_yes(row, split="val"):
    model.eval()

    image, img_path = load_image(row, split)
    choices = parse_choices(row.get("choices", ""))

    if len(choices) == 0:
        return 0, [], img_path

    prompts = []

    for i, choice in enumerate(choices):
        letter = LETTERS[i]
        choice_text = f"{letter}. {str(choice).strip()}"
        prompts.append(build_pref_prompt(row, choice_text))

    texts = [make_chat_text(p) for p in prompts]
    images = [image for _ in prompts]

    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    model_device = next(model.parameters()).device
    inputs = {
        k: v.to(model_device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    out = model(**inputs)
    logits = out.logits[:, -1, :]
    scores = yes_logprob_from_logits(logits)

    pred = int(torch.argmax(scores).item())

    return pred, scores.detach().cpu().tolist(), img_path


@torch.no_grad()
def evaluate_val_real_once():
    model.eval()

    rows = []
    correct = 0
    image_correct = 0
    image_total = 0
    text_correct = 0
    text_total = 0

    for idx, row in tqdm(val_df.iterrows(), total=len(val_df), desc="REAL val once"):
        y = int(row["answer"])
        pred, scores, img_path = score_row_yes(row, split="val")

        ok = int(pred == y)
        correct += ok

        img_heavy = is_image_heavy(row)

        if img_heavy:
            image_correct += ok
            image_total += 1
        else:
            text_correct += ok
            text_total += 1

        rows.append({
            "idx": idx,
            "id": row.get("id", idx),
            "answer": y,
            "pred": pred,
            "correct": ok,
            "scores": scores,
            "image_heavy": img_heavy,
            "image_path": str(img_path) if img_path else "",
        })

    acc = correct / max(len(val_df), 1)
    image_acc = image_correct / max(image_total, 1)
    text_acc = text_correct / max(text_total, 1)

    val_pred_df = pd.DataFrame(rows)
    model.train()

    return {
        "acc": acc,
        "image_heavy_acc": image_acc,
        "text_acc": text_acc,
        "pred_df": val_pred_df,
    }


optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

total_update_steps = math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS
warmup_steps = int(total_update_steps * WARMUP_RATIO)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, total_update_steps - warmup_steps)
    return max(0.05, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

best_acc = -1.0
history = []
global_step = 0

print("Starting cached-reference DPO training...")
print("Total update steps:", total_update_steps)
print("Warmup steps:", warmup_steps)

for epoch in range(1, EPOCHS + 1):
    model.train()

    epoch_start = time.time()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader, start=1):
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            loss = preference_loss(batch) / GRAD_ACCUM

        scaler.scale(loss).backward()
        running_loss += loss.item() * GRAD_ACCUM

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad],
                MAX_GRAD_NORM
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

            scheduler.step()
            global_step += 1

        if step % PRINT_EVERY == 0:
            elapsed = (time.time() - epoch_start) / 60
            steps_per_min = step / max(elapsed, 1e-6)
            eta = (len(train_loader) - step) / max(steps_per_min, 1e-6)
            avg_loss = running_loss / step

            print(
                f"Epoch {epoch}/{EPOCHS} | step {step}/{len(train_loader)} "
                f"| {LOSS_TYPE.upper()} loss {avg_loss:.4f} "
                f"| lr {scheduler.get_last_lr()[0]:.2e} "
                f"| elapsed {elapsed:.1f}m | ETA {eta:.1f}m"
            )

    epoch_min = (time.time() - epoch_start) / 60
    avg_loss = running_loss / max(len(train_loader), 1)

    print(f"\nEpoch {epoch} finished in {epoch_min:.2f} min | avg loss {avg_loss:.4f}")

    print("Running REAL-image validation once...")
    val_result = evaluate_val_real_once()

    val_pred_path = OUT_DIR / f"val_predictions_epoch_{epoch}.csv"
    val_result["pred_df"].to_csv(val_pred_path, index=False)

    row_hist = {
        "epoch": epoch,
        "epoch_min": epoch_min,
        "loss": avg_loss,
        "real_acc": val_result["acc"],
        "real_image_heavy_acc": val_result["image_heavy_acc"],
        "real_text_acc": val_result["text_acc"],
    }

    history.append(row_hist)
    pd.DataFrame(history).to_csv(OUT_DIR / "training_history_continue.csv", index=False)

    print("\nEPOCH SUMMARY")
    for k, v in row_hist.items():
        print(k, ":", v)

    model.save_pretrained(LAST_DIR)
    processor.save_pretrained(LAST_DIR)

    if val_result["acc"] > best_acc:
        best_acc = val_result["acc"]
        model.save_pretrained(BEST_DIR)
        processor.save_pretrained(BEST_DIR)
        print("Saved NEW BEST adapter:", BEST_DIR, "acc:", best_acc)
    else:
        print("No improvement. Best remains:", best_acc)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nTRAINING DONE")
print("Best real-image val acc:", best_acc)
print("Best adapter:", BEST_DIR)

display(pd.DataFrame(history))

print("\nReloading best adapter for test inference...")

try:
    del model
    del base_policy
except Exception:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

MAX_IMAGE_SIZE = 448
print("Test MAX_IMAGE_SIZE:", MAX_IMAGE_SIZE)

base_test = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
)

model = PeftModel.from_pretrained(
    base_test,
    BEST_DIR,
    is_trainable=False
)

model.eval()

pred_rows = []
t0 = time.time()

print("Running test inference...")

for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
    try:
        pred, scores, img_path = score_row_yes(row, split="test")

        choices = parse_choices(row.get("choices", ""))
        if pred >= len(choices):
            pred = 0

        pred_rows.append({
            "id": row["id"],
            "answer": int(pred),
            "scores": scores,
            "num_choices": len(choices),
            "qtype": detect_question_type(row),
            "image_heavy": is_image_heavy(row),
            "image_path": str(img_path) if img_path else "",
            "error": "",
        })

    except Exception as e:
        pred_rows.append({
            "id": row["id"],
            "answer": 0,
            "scores": [],
            "num_choices": 0,
            "qtype": "",
            "image_heavy": False,
            "image_path": "",
            "error": repr(e),
        })

    if (idx + 1) % 50 == 0:
        elapsed = (time.time() - t0) / 60
        print(f"Done {idx+1}/{len(test_df)} | elapsed {elapsed:.1f}m")

    if (idx + 1) % 100 == 0:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

pred_df = pd.DataFrame(pred_rows)

submission = pred_df[["id", "answer"]].copy()
submission["answer"] = submission["answer"].astype(int)

submission.to_csv(SUB_PATH, index=False)
pred_df.to_csv(DEBUG_PATH, index=False)

print("\nDONE.")
print("Saved submission:", SUB_PATH)
print("Saved debug:", DEBUG_PATH)
print("Elapsed minutes:", (time.time() - t0) / 60)

print("\nSubmission preview:")
display(submission.head())

print("\nAnswer distribution:")
display(submission["answer"].value_counts().sort_index())

print("\nQuestion type distribution:")
display(pred_df["qtype"].value_counts())

print("\nErrors:")
display(pred_df[pred_df["error"] != ""].head(20))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train: (3109, 15)
val: (1048, 15)
test: (1008, 13)
cached ref rows: 9666
ref lookup size: 9666


chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

,count
answer,
0,372
1,373
2,192
3,71



Question type distribution:


,count
qtype,
diagram,399
table,201
general,174
graph,116
map,65
punnett,35
passage,14
food_web,3
circuit,1



Errors:


,id,answer,scores,num_choices,qtype,image_heavy,image_path,error
